# Homework-4
Recurrent Neural Networks  

## Problem 1:

Answer each of the following questions.

(A) Both CNNs and RNNs share parameters—but in different ways. Explain.

> Both CNNs and RNNs apply filters/kernels to the data, but across different dimensions. A CNN will apply the filter across space, scanning through every pixel position in an image. Meanwhile, RNNs will apply the filter across time, scanning across every time step in the sample. 


(B) What is the basic idea underlying how we learn meaningful word embeddings? How does this relate to the idea of self-supervised learning?

> We learn meaningful word embeddings from the context in which they are found. When we train a model, we have it predict words based on the context which allow the internal representations that the model develops to solve that task and encode semantic structure. As such, the data becomes self-supervised since the training signal comes from the data itself rather than a person labelling it by hand.

(C) Consider an LSTM layer that accepts a batch of arbitrary size, with 40 time steps and 3 features. The LSTM layer has input shape `[None, 40, 3]`. Suppose its output is shape `[None, 2]`. Find the number of parameters in the LSTM layer, explaining your thinking.

> In an LSTM, there are four gates (input, forget, output, and cell). Within each gate, there is are input weights that is 2x3, hidden weights that are 2x2, and a bias term that is 2x1. Adding these all up there are 12 parameters in each gate, times 4 we get 48 parameters in one LSTM layer. 

## Problem-2: 

 Begin by reviewing Andrej Karpathy's famous blog post [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/).

Provide a three-paragraph summary of the article, describing Karpathy's main points.

INSERT ANSWER HERE

## Problem-3: 

Write code for a character-based RNN (i.e. LSTM or GRU, not simple_RNN) in PyTorch. You can use his source code and any code online to assist. If you do, just reference where you sought help from.

- Choose a large text corpus, such as a collection of novels from [Project Gutenburg](https://www.gutenberg.org/). You can use other data, but you should explain where your data comes from.

> I chose to download every novel by Fyodor Dostoyevsky

- Perform any necessary preprocessing, explaining what steps you take. In particular, what forms of normalization do you use? Do you define characters with special meaning?

In [1]:
# Load packages
import pandas as pd 
import nltk 
import re
from pathlib import Path

import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset, random_split

In [2]:
# Function to remove header
def clean_gutenberg(text):
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
    
    start_idx = text.find(start_marker)
    end_idx = text.find(end_marker)
    
    if start_idx != -1:
        # Move past the marker line itself
        start_idx = text.find("\n", start_idx) + 1
    else:
        start_idx = 0  
    
    if end_idx != -1:
        text = text[start_idx:end_idx]
    else:
        text = text[start_idx:] 
    
    return text.strip()

# Import books 
text = ""
for filepath in Path('dostoyevsky/').glob('*.txt'):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()
    
    cleaned = clean_gutenberg(raw)
    text += cleaned + "\n"

print(f"\nTotal characters: {len(text):,}")

### Pre-Processing 
text = text.lower()
# text = re.sub(r'[^\x00-\x7F]', '', text)

## Build character vocabulary
chars = sorted(set(text)) 
data_size, vocab_size = len(text), len(chars)
print(f'Data has {data_size} characters, {vocab_size} unique characters.')

## Mappings
char_to_ix = {ch:i for i, ch in enumerate(chars)}
ix_to_char = {i:ch for i, ch in enumerate(chars)}

## Encode text as integers
encoded = [char_to_ix[ch] for ch in text]


Total characters: 4,712,707
Data has 4712707 characters, 73 unique characters.


> The only normalization steps are converting the text to lower case
> 
> No special characters were defined. The vocabulary consists entirely of characters that appear naturally in the text after normalization including lowercase ASCII letters, digits, punctuation, and whitespace. Newline characters (\n) are left in the vocabulary and carry implicit meaning as paragraph/line boundaries.

- Efficiently load and batch the dataset for training using a `DataLoader`. Make sure to reserve some of the data for validation and testing. Describe how you handle batching and sequence lengths.

In [3]:
## Create input/target sequences
seq_length = 100  

inputs, targets = [], []
for i in range(len(encoded) - seq_length):
    inputs.append(encoded[i : i + seq_length])
    targets.append(encoded[i + 1 : i + seq_length + 1]) 

# Load and batch
X = torch.tensor(inputs, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)
dataset = TensorDataset(X, y)

# Split data
total = len(dataset)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

# Load data
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

> The batch size is set to 64, meaning the model sees 64 sequences at once per training step rather than one at a time. The shuffle=True option means batches are randomly assembled each epoch, which helps the model generalize. 
>
> The sequence length is set to 100, meaning every input sequence is exactly 100 characters long. This ensures all inputs are the same length, eliminating the need for padding. 


- Define your RNN model. Discuss the number of layers, hidden units, and the type of RNN cells you use. What is the total number of parameters in your model? Explain the rationale behind your architectural choices.

In [4]:
# Define LSTM class
class LSTMModel(nn.Module):
    def __init__(self, vocab_dim, embed_dim, hidden_dim, layer_dim, dropout=0.5):
        super(LSTMModel, self).__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.embedding = nn.Embedding(vocab_dim,embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, layer_dim, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(self.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(self.device)
        
        x = self.embedding(x)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out)
        return out, hn, cn

In [7]:
# CPU/GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:",device)

# Intialize model, loss function, and optimizer
model = LSTMModel(vocab_dim  = vocab_size, 
                  embed_dim  = 64, 
                  hidden_dim = 256, 
                  layer_dim  = 2,
                  dropout    = 0.5
                  ).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

Device: cuda


> I chose a LSTM model for this exercise since they are typically more accurate than a vanilla RNN. 
> 
> Source code: https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/ 


- Write the training loop.

In [ ]:
# INSERT CODE


- Monitor and report on the training progress by tracking the loss. After training, evaluate the model's performance using a suitable evaluation metric (e.g., perplexity) on a validation dataset or a held-out portion of the training data. Discuss the results.

In [ ]:
# INSERT CODE


- Specify the hyper-parameters (for example, model hyper-parameters, as well as sampling size and beam width from below) used in your model. Find suitable settings using a validation split.

In [ ]:
# INSERT CODE


- Implement a text generation function using the trained RNN model. Provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.

In [ ]:
# INSERT CODE

## Problem-4: Optional extra credit 
 
Up to +3 bonus points 

* Repeat the text generation process from the previous problem, but do it with a transformer architecture rather than an LSTM/GRU (you can use word tokens instead of character tokens if you prefer). Use the provided mini-GPT lab for reference.  

## Problem-5: Optional extra credit 
 


- (+1 bonus points): Now with your model trained, implement top-k and nucleus sampling. Again, provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.

In [ ]:
# INSERT CODE

- (+1 bonus points): Now with your model trained, implement beam search. Again, provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.  

In [ ]:
# INSERT CODE